# Ollama on Google Colab (Cloudflare Tunnel)

Run this notebook to set up Ollama on Colab and connect it to your Exam prep app.
After running, copy the Cloudflare Tunnel URL and set `OLLAMA_BASE_URL` in your `.env.local`.

In [ ]:
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Start Ollama server in background
import subprocess
import time

subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
print('Ollama server started')

In [ ]:
# Pull a model (gemma2:2b is lightweight and good for Thai/English)
# You can change to other models: qwen2.5:1.5b, llama3.2:3b, etc.
MODEL_NAME = "gemma2:2b"  # @param {type:"string"}
!ollama pull $MODEL_NAME

In [ ]:
# Test the model
!ollama run $MODEL_NAME "Generate a multiple-choice Thai exam question about percentage"

In [ ]:
# Install cloudflared (Cloudflare Tunnel client)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print('cloudflared installed')

In [ ]:
# Start Cloudflare Tunnel to expose Ollama API (port 11434)
import subprocess
import threading
import re
import time

tunnel_url = None

def run_tunnel():
    global tunnel_url
    proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://localhost:11434', '--no-autoupdate'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        universal_newlines=True
    )
    for line in proc.stdout:
        print(line, end='')
        m = re.search(r'https://[a-zA-Z0-9.-]+\.trycloudflare\.com', line)
        if m:
            tunnel_url = m.group(0)
            print(f"\n*** TUNNEL URL: {tunnel_url} ***")

t = threading.Thread(target=run_tunnel, daemon=True)
t.start()
time.sleep(8)

if tunnel_url:
    print(f"\n{'='*60}")
    print(f"Ollama API is accessible at:")
    print(f"  {tunnel_url}/api")
    print(f"\nCopy this URL and set it as OLLAMA_BASE_URL in your .env.local:")
    print(f"  OLLAMA_BASE_URL={tunnel_url}/api")
    print(f"  OLLAMA_MODEL={MODEL_NAME}")
    print(f"  ENABLE_OLLAMA=1")
    print(f"{'='*60}")
else:
    print("Waiting for tunnel URL...")
    time.sleep(5)
    if tunnel_url:
        print(f"OLLAMA_BASE_URL={tunnel_url}/api")
    else:
        print("Tunnel URL not found yet. Check output above for the trycloudflare.com URL.")

In [ ]:
# Quick test: call the Ollama API through Cloudflare Tunnel
import requests

if tunnel_url:
    ollama_api = f"{tunnel_url}/api/chat"
    response = requests.post(ollama_api, json={
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": "Say hello in Thai"}],
        "stream": False
    }, timeout=30)
    print(response.json()["message"]["content"])
else:
    print("Tunnel URL not available yet. Run the previous cell again.")

## Keep this notebook running

**Do not close this notebook** while you're using the app. The Cloudflare Tunnel will stop when the Colab runtime disconnects.

### Troubleshooting
- If the tunnel URL doesn't appear after 15s, re-run the tunnel cell.
- Cloudflare Tunnel is free and doesn't require an account.
- To update the model: `!ollama pull <new-model>` and update `OLLAMA_MODEL` in `.env.local`.